# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saibalaji2004/Flyrank-intern-repo/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions and reason codes

The action queue uses the existing Week-4 rule-based baseline because its rules are transparent and easy for a content strategist to explain. It gives priority to pages that are both stale and visible, followed by pages with only one of those signals.

| Priority score | Action | Reason code | Recommended next step |
|---|---|---|---|
| 3 | `REVIEW_CONTENT` | `STALE_AND_VISIBLE` | Review search intent, content accuracy, coverage, and competing results before deciding whether to refresh. |
| 2 | `REVIEW_STALENESS` | `STALE_ONLY` | Check whether the page is outdated or needs maintenance. Confirm that updating it would benefit users. |
| 1 | `REVIEW_VISIBILITY` | `VISIBLE_ONLY` | Investigate impressions, click-through rate, search intent, and page relevance before changing content. |
| 0 | `MONITOR` | `NEITHER_SIGNAL` | Keep the page under observation rather than prioritizing it for immediate review. |

A page receives two points for being at least 90 days since its last update and one point for having impressions at or above the baseline threshold of 731 impressions over 90 days. The queue is sorted by priority score, then update age and impressions.

These are review priorities, not proof that a page is declining or that refreshing it will improve search performance. The Logistic Regression model showed better Precision@10 and Precision@20 than the baseline on the held-out client split, but its ROC-AUC of 0.573 indicates limited overall discrimination. Human review remains necessary.

In [6]:

import pandas as pd
import numpy as np
from pathlib import Path

# Load the anonymized starter dataset.
dataset_url = (
    "https://raw.githubusercontent.com/"
    "saibalaji2004/Flyrank-intern-repo/main/"
    "data/raw/content_refresh_anonymized.csv"
)
df = pd.read_csv(dataset_url)

required_columns = {
    "content_id",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction",
}
missing_columns = required_columns - set(df.columns)
assert not missing_columns, f"Missing columns: {missing_columns}"

# Reproduce the fixed Week-4 baseline rules.
positive_impressions = df.loc[
    df["impressions_90d"] > 0, "impressions_90d"
]
visibility_threshold = positive_impressions.median()

df["is_stale"] = (
    df["days_since_last_update"].ge(90).fillna(False)
)
df["is_visible"] = (
    df["impressions_90d"].ge(visibility_threshold).fillna(False)
)

df["priority_score"] = (
    2 * df["is_stale"].astype(int)
    + df["is_visible"].astype(int)
)

action_map = {
    3: "REVIEW_CONTENT",
    2: "REVIEW_STALENESS",
    1: "REVIEW_VISIBILITY",
    0: "MONITOR",
}

reason_map = {
    3: "STALE_AND_VISIBLE",
    2: "STALE_ONLY",
    1: "VISIBLE_ONLY",
    0: "NEITHER_SIGNAL",
}

df["recommended_action"] = df["priority_score"].map(action_map)
df["reason_code"] = df["priority_score"].map(reason_map)

# Keep missing ranking values last, matching the baseline's intent.
df["rank_days_stale"] = df["days_since_last_update"].fillna(-1)
df["rank_impressions"] = df["impressions_90d"].fillna(-1)

action_queue = df.sort_values(
    by=[
        "priority_score",
        "rank_days_stale",
        "rank_impressions",
    ],
    ascending=[False, False, False],
    kind="mergesort",
).copy()

action_queue["queue_rank"] = np.arange(1, len(action_queue) + 1)

# Check action and reason-code summaries.
print("Rows in queue:", len(action_queue))
print("Visibility threshold:", visibility_threshold)

print("\nRecommended action counts:")
print(action_queue["recommended_action"].value_counts())

print("\nReason-code counts:")
print(action_queue["reason_code"].value_counts())

print("\nTop 10 priority scores and reason codes:")
print(
    action_queue[
        ["queue_rank", "priority_score",
         "recommended_action", "reason_code"]
    ].head(10).to_string(index=False)
)

# Integrity checks.
assert len(action_queue) == len(df)
assert action_queue["content_id"].is_unique
assert action_queue["recommended_action"].notna().all()
assert action_queue["reason_code"].notna().all()
assert action_queue["priority_score"].between(0, 3).all()

print("\nPASS: action queue and reason-code checks completed.")
print("Note: this is a review queue, not a causal or future-performance claim.")


Rows in queue: 30000
Visibility threshold: 731.0

Recommended action counts:
recommended_action
MONITOR              11640
REVIEW_VISIBILITY     9015
REVIEW_CONTENT        5992
REVIEW_STALENESS      3353
Name: count, dtype: int64

Reason-code counts:
reason_code
NEITHER_SIGNAL       11640
VISIBLE_ONLY          9015
STALE_AND_VISIBLE     5992
STALE_ONLY            3353
Name: count, dtype: int64

Top 10 priority scores and reason codes:
 queue_rank  priority_score recommended_action       reason_code
          1               3     REVIEW_CONTENT STALE_AND_VISIBLE
          2               3     REVIEW_CONTENT STALE_AND_VISIBLE
          3               3     REVIEW_CONTENT STALE_AND_VISIBLE
          4               3     REVIEW_CONTENT STALE_AND_VISIBLE
          5               3     REVIEW_CONTENT STALE_AND_VISIBLE
          6               3     REVIEW_CONTENT STALE_AND_VISIBLE
          7               3     REVIEW_CONTENT STALE_AND_VISIBLE
          8               3     REVIEW_CO

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

The action playbook is intended for content strategists and SEO teams who need to prioritize pages for manual review. It uses page freshness and 90-day impression volume to produce a transparent, explainable review queue.

Pages marked `REVIEW_CONTENT` should be examined first because they meet both baseline rules: at least 90 days since the last update and impressions at or above the threshold of 731. The other actions help teams investigate staleness or visibility individually, while `MONITOR` identifies pages that meet neither rule.

This is a decision-support tool, not an automatic content-refresh system. The rules do not directly measure content quality, search intent satisfaction, business value, or the cause of declining performance. The dataset's decline label is derived from `trend_direction`, so it is a proxy for observed decline rather than confirmation that a page needs refreshing.

The Logistic Regression model achieved Precision@10 of 0.60 and Precision@20 of 0.65 on the held-out client split, compared with 0.40 and 0.40 for the rule-based baseline. However, its ROC-AUC of 0.573 indicates limited overall discrimination. These results are evidence from this dataset and split, not a guarantee of future performance. The dataset also lacks the complete point-in-time history needed to establish reliable future-period performance or causal effects from content updates.

In [7]:

# Section 2: intended-use and limitation checks

expected_actions = {
    "REVIEW_CONTENT",
    "REVIEW_STALENESS",
    "REVIEW_VISIBILITY",
    "MONITOR",
}

assert set(action_queue["recommended_action"].unique()) == expected_actions
assert visibility_threshold == 731.0

# Confirm the review queue's rules are applied consistently.
assert (
    action_queue.loc[
        action_queue["priority_score"] == 3, "is_stale"
    ].all()
)
assert (
    action_queue.loc[
        action_queue["priority_score"] == 3, "is_visible"
    ].all()
)

# Record the previously measured held-out evaluation results.
evaluation_summary = {
    "baseline_precision_at_10": 0.40,
    "model_precision_at_10": 0.60,
    "baseline_precision_at_20": 0.40,
    "model_precision_at_20": 0.65,
    "model_roc_auc": 0.5726,
}

print("Intended users: content strategists and SEO teams")
print("Purpose: prioritize pages for human review")
print("Action categories:", sorted(expected_actions))

print("\nPreviously measured held-out model evaluation:")
for metric, value in evaluation_summary.items():
    print(f"{metric}: {value:.4f}")

print("\nLimitations:")
print("- The decline label is a proxy derived from trend_direction.")
print("- The baseline rules do not establish content quality or causality.")
print("- Held-out results do not guarantee future performance.")
print("- Human review is required before taking action.")

print("\nPASS: intended-use and limitation checks completed.")


Intended users: content strategists and SEO teams
Purpose: prioritize pages for human review
Action categories: ['MONITOR', 'REVIEW_CONTENT', 'REVIEW_STALENESS', 'REVIEW_VISIBILITY']

Previously measured held-out model evaluation:
baseline_precision_at_10: 0.4000
model_precision_at_10: 0.6000
baseline_precision_at_20: 0.4000
model_precision_at_20: 0.6500
model_roc_auc: 0.5726

Limitations:
- The decline label is a proxy derived from trend_direction.
- The baseline rules do not establish content quality or causality.
- Held-out results do not guarantee future performance.
- Human review is required before taking action.

PASS: intended-use and limitation checks completed.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review and the no-go list

Every recommendation must be reviewed by a content strategist before action is taken. The reviewer should confirm the page's purpose, target search intent, factual accuracy, freshness, user value, and current search performance. They should also examine whether changes in impressions or clicks could reflect seasonality, demand changes, tracking issues, or other factors rather than outdated content.

For `REVIEW_CONTENT`, check whether the page needs meaningful improvements to accuracy, completeness, relevance, or usefulness. For `REVIEW_STALENESS`, verify whether the content is genuinely outdated. For `REVIEW_VISIBILITY`, investigate impressions, click-through rate, search intent, and relevance before deciding on a change. For `MONITOR`, continue observing performance rather than assuming the page is healthy.

The score and reason code explain why a page entered the queue; they do not prove that the recommended action is correct. Reviewers should record their decision and rationale, and avoid making unnecessary edits merely to change a page's freshness date.

**No-go list:** Do not automatically rewrite, publish, delete, redirect, or de-index pages based only on a score. Do not guarantee ranking improvements, infer causation from the decline label, expose client identifiers or private search data, or use the queue as a substitute for editorial judgment. High-impact changes require appropriate human approval and post-change monitoring.

In [8]:

# Section 3: human-review and no-go policy checks

review_policy = {
    "human_approval_required": True,
    "automatic_publishing_allowed": False,
    "automatic_deletion_allowed": False,
    "automatic_redirects_allowed": False,
    "ranking_improvement_guaranteed": False,
    "client_identifiers_in_public_queue": False,
}

for policy, value in review_policy.items():
    print(f"{policy}: {value}")

assert review_policy["human_approval_required"]
assert not review_policy["automatic_publishing_allowed"]
assert not review_policy["automatic_deletion_allowed"]
assert not review_policy["automatic_redirects_allowed"]
assert not review_policy["ranking_improvement_guaranteed"]
assert not review_policy["client_identifiers_in_public_queue"]

# The shared queue should contain no client ID field.
public_queue_columns = {
    "queue_rank",
    "priority_score",
    "recommended_action",
    "reason_code",
}
assert public_queue_columns.issubset(action_queue.columns)
assert "client_id" not in public_queue_columns
assert "content_id" not in public_queue_columns

print("\nPASS: human-review policy and public-output checks completed.")


human_approval_required: True
automatic_publishing_allowed: False
automatic_deletion_allowed: False
automatic_redirects_allowed: False
ranking_improvement_guaranteed: False
client_identifiers_in_public_queue: False

PASS: human-review policy and public-output checks completed.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retraining triggers

The action playbook should be reviewed regularly to check whether its recommendations remain useful. Monitor the distribution of priority scores and reason codes, the amount of missing data, changes in input-feature distributions, and the proportion of reviewed pages that are confirmed to need attention.

After a content change, compare impressions, clicks, click-through rate, and average position before and after the change over comparable observation windows. Account for seasonality, demand changes, and tracking issues. These comparisons can show whether performance changed, but they do not prove that the content change caused the result.

Reassess the baseline threshold and scoring rules when the data distribution changes substantially, the queue becomes dominated by one action category, or content strategists repeatedly disagree with its recommendations. Retrain and re-evaluate the Logistic Regression model when enough newly labelled examples are available, feature definitions change, or measured performance deteriorates on an appropriate held-out evaluation.

Before deployment, compare the updated approach with the existing baseline using the same evaluation population and metrics, and confirm that no prohibited target-derived fields enter the feature set. Since this dataset lacks a complete point-in-time history, it cannot by itself establish future-period performance. Monitoring thresholds and review frequency should be defined with the team before operational use.

In [9]:

# Section 4: monitoring indicators and retraining triggers

monitoring_summary = {
    "total_pages": len(action_queue),
    "missing_last_update_days": int(
        action_queue["days_since_last_update"].isna().sum()
    ),
    "missing_90d_impressions": int(
        action_queue["impressions_90d"].isna().sum()
    ),
    "priority_score_3_count": int(
        action_queue["priority_score"].eq(3).sum()
    ),
    "priority_score_2_count": int(
        action_queue["priority_score"].eq(2).sum()
    ),
    "priority_score_1_count": int(
        action_queue["priority_score"].eq(1).sum()
    ),
    "priority_score_0_count": int(
        action_queue["priority_score"].eq(0).sum()
    ),
}

print("Current queue monitoring snapshot:")
for metric, value in monitoring_summary.items():
    print(f"{metric}: {value}")

action_distribution = (
    action_queue["recommended_action"].value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nAction distribution (%):")
print(action_distribution.to_string())

monitoring_triggers = [
    "Substantial change in input-feature distributions",
    "Increase in missing or invalid input data",
    "Queue distribution becomes heavily concentrated in one action",
    "Human reviewers frequently disagree with recommendations",
    "Held-out evaluation metrics deteriorate",
    "New labelled data or changed feature definitions become available",
]

print("\nTriggers for investigation or retraining:")
for trigger in monitoring_triggers:
    print("-", trigger)

assert monitoring_summary["total_pages"] == 30000
assert sum(
    monitoring_summary[key]
    for key in [
        "priority_score_3_count",
        "priority_score_2_count",
        "priority_score_1_count",
        "priority_score_0_count",
    ]
) == len(action_queue)

print("\nPASS: monitoring snapshot and trigger checks completed.")
print("Note: these are monitoring indicators, not proof of model drift.")


Current queue monitoring snapshot:
total_pages: 30000
missing_last_update_days: 0
missing_90d_impressions: 0
priority_score_3_count: 5992
priority_score_2_count: 3353
priority_score_1_count: 9015
priority_score_0_count: 11640

Action distribution (%):
recommended_action
MONITOR              38.80
REVIEW_VISIBILITY    30.05
REVIEW_CONTENT       19.97
REVIEW_STALENESS     11.18

Triggers for investigation or retraining:
- Substantial change in input-feature distributions
- Increase in missing or invalid input data
- Queue distribution becomes heavily concentrated in one action
- Human reviewers frequently disagree with recommendations
- Held-out evaluation metrics deteriorate
- New labelled data or changed feature definitions become available

PASS: monitoring snapshot and trigger checks completed.
Note: these are monitoring indicators, not proof of model drift.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the paper

The notebook exports a ranked action queue and an aggregate action summary for use in the research paper. The queue contains only the rank, priority score, recommended action, reason code, and the two signals used by the baseline. It excludes content IDs, client IDs, private URLs, and search queries.

The aggregate summary reports the number and percentage of pages assigned to each action. These files document the current rule-based review queue; they do not establish that the recommendations are correct, predict future performance, or prove that a content refresh causes search improvements.

In [10]:

# Section 5: export privacy-safe paper artifacts

from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Export only fields needed to explain and reproduce the queue.
safe_columns = [
    "queue_rank",
    "priority_score",
    "recommended_action",
    "reason_code",
    "days_since_last_update",
    "impressions_90d",
]

paper_queue = action_queue[safe_columns].copy()

# Aggregate summary for the paper.
action_summary = (
    action_queue["recommended_action"]
    .value_counts()
    .rename_axis("recommended_action")
    .reset_index(name="page_count")
)
action_summary["percentage"] = (
    action_summary["page_count"] / len(action_queue) * 100
).round(2)

queue_path = output_dir / "ml10_ranked_action_queue.csv"
summary_path = output_dir / "ml10_action_summary.csv"

paper_queue.to_csv(queue_path, index=False)
action_summary.to_csv(summary_path, index=False)

# Verify files and privacy-sensitive column exclusions.
queue_check = pd.read_csv(queue_path)
summary_check = pd.read_csv(summary_path)

for forbidden_column in [
    "content_id",
    "client_id",
    "url",
    "query",
]:
    assert forbidden_column not in queue_check.columns

assert len(queue_check) == len(action_queue)
assert summary_check["page_count"].sum() == len(action_queue)
assert queue_check["queue_rank"].is_unique
assert queue_check["recommended_action"].notna().all()

print("Exported files:")
print("-", queue_path)
print("-", summary_path)

print("\nRanked queue shape:", queue_check.shape)
print("\nAction summary:")
print(summary_check.to_string(index=False))

print("\nPASS: export, row-count, ranking, and column checks completed.")


Exported files:
- work/outputs/ml10_ranked_action_queue.csv
- work/outputs/ml10_action_summary.csv

Ranked queue shape: (30000, 6)

Action summary:
recommended_action  page_count  percentage
           MONITOR       11640       38.80
 REVIEW_VISIBILITY        9015       30.05
    REVIEW_CONTENT        5992       19.97
  REVIEW_STALENESS        3353       11.18

PASS: export, row-count, ranking, and column checks completed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.